# SongNet: Real-time Music Genre Classification (UE24CS352A Mini-Project, Problem #33)

Reproduction of the Stanford CS229 (2018) SongNet paper on **FMA-small** (8 genres), with classical baselines, a 1D-conv SongNet, a GRU-head extension, and two data splits.

**How to use this notebook**
1. `File > Save a copy in Drive` so you have your own copy.
2. `Runtime > Change runtime type > T4 GPU`.
3. Run the cells **in order**. Cells marked *(slow)* take a while; don't close the tab while they run.
4. Colab deletes its files when the session ends, so run the **backup** cells at the end.

**Rough timings on a T4 (estimates):** download ~7 min, preprocessing ~13 min, baselines ~5-10 min each split, each training run ~15-40 min.

**Splits used**
- *Official* FMA split (80/10/10): stricter test.
- *Random* 70/20/10 split: the protocol described in the paper.

Always compare a model against baselines **on the same split**.

## 1. Check the GPU

In [1]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())

Sat Oct  3 17:58:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install dependencies and create folders

In [ ]:
!pip install -q librosa joblib tqdm seaborn scikit-learn
!mkdir -p src data results results_random

## 3. Write the project source files
These cells create `src/*.py`. Run them all once.

In [ ]:
%%writefile src/data.py
"""Convert FMA-small mp3s into fixed-size log-mel spectrograms.

Usage:
  python src/data.py --meta data/fma_metadata --audio data/fma_small --out data/processed
Uses FMA's official split (training / validation / test = 80/10/10).
"""
import argparse, json
from pathlib import Path
import numpy as np, pandas as pd, librosa
from joblib import Parallel, delayed
from tqdm import tqdm

SR, N_FFT, HOP, N_MELS, FRAMES = 22050, 2048, 1024, 128, 640  # ~29.7 s


def audio_to_mel(y):
    """Waveform -> log-mel (N_MELS, T). Shared with the real-time demo."""
    m = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS)
    return librosa.power_to_db(m, ref=1.0).astype(np.float32)


def fix_length(m):
    if m.shape[1] < FRAMES:
        m = np.pad(m, ((0, 0), (0, FRAMES - m.shape[1])), constant_values=m.min())
    return m[:, :FRAMES]


def load_one(path):
    try:
        y, _ = librosa.load(path, sr=SR, duration=30)
        return fix_length(audio_to_mel(y)).astype(np.float16)
    except Exception:
        return None  # a few FMA mp3s are corrupt; they get skipped


def track_path(audio_dir, tid):
    s = f"{tid:06d}"
    return str(Path(audio_dir) / s[:3] / f"{s}.mp3")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--meta", required=True)
    ap.add_argument("--audio", required=True)
    ap.add_argument("--out", default="data/processed")
    ap.add_argument("--jobs", type=int, default=-1)
    a = ap.parse_args()

    tracks = pd.read_csv(Path(a.meta) / "tracks.csv", index_col=0, header=[0, 1])
    small = tracks[tracks["set", "subset"] == "small"]
    classes = sorted(small["track", "genre_top"].dropna().unique())
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    json.dump(classes, open(out / "classes.json", "w"))

    for split in ["training", "validation", "test"]:
        df = small[small["set", "split"] == split]
        ids = df.index.tolist()
        mels = Parallel(n_jobs=a.jobs)(
            delayed(load_one)(track_path(a.audio, t)) for t in tqdm(ids, desc=split))
        keep = [i for i, m in enumerate(mels) if m is not None]
        X = np.stack([mels[i] for i in keep])
        y = np.array([classes.index(df["track", "genre_top"].iloc[i]) for i in keep])
        np.save(out / f"{split}_X.npy", X); np.save(out / f"{split}_y.npy", y)
        np.save(out / f"{split}_ids.npy", np.array([ids[i] for i in keep]))
        print(split, X.shape, f"skipped {len(ids) - len(keep)}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/model.py
import torch, torch.nn as nn, torch.nn.functional as F


class SongNet(nn.Module):
    """1D-conv over time (mel bins = channels) -> per-timestep genre logits.
    head='dense': time-distributed linear layer (as in the paper)
    head='gru'  : causal GRU before the linear layer (real recurrent head)
    Song-level prediction = mean of per-timestep probabilities.
    """
    def __init__(self, n_mels=128, n_classes=8, channels=(128, 256, 256), k=5, p=0.25, head="dense"):
        super().__init__()
        layers, c = [], n_mels
        for o in channels:
            layers += [nn.Conv1d(c, o, k, padding=k // 2), nn.BatchNorm1d(o),
                       nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(p)]
            c = o
        self.cnn = nn.Sequential(*layers)
        self.head = head
        if head == "gru":
            self.rnn = nn.GRU(c, 256, batch_first=True)  # unidirectional -> usable in real time
            c = 256
        self.fc = nn.Linear(c, n_classes)

    def forward(self, x):                 # x: (B, n_mels, T)
        h = self.cnn(x).transpose(1, 2)   # (B, T', C)
        if self.head == "gru":
            h, _ = self.rnn(h)
        return self.fc(h)                 # (B, T', n_classes)


def song_log_probs(logits):
    """Mean of per-timestep softmax -> log-probabilities per song."""
    return torch.log(F.softmax(logits, -1).mean(1) + 1e-8)

In [ ]:
%%writefile src/train.py
"""Train + evaluate SongNet (v2: random time crops, SpecAugment-style masking,
per-mel-bin normalization, AdamW).
Usage: python src/train.py --data data/processed --head dense
"""
import argparse, json
from pathlib import Path
import numpy as np, torch, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg")
import seaborn as sns, matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from model import SongNet, song_log_probs

ap = argparse.ArgumentParser()
ap.add_argument("--data", default="data/processed")
ap.add_argument("--head", choices=["dense", "gru"], default="dense")
ap.add_argument("--epochs", type=int, default=80)
ap.add_argument("--bs", type=int, default=32)
ap.add_argument("--lr", type=float, default=1e-3)
ap.add_argument("--wd", type=float, default=1e-2)
ap.add_argument("--crop", type=int, default=320, help="training crop length in frames (640 = full clip)")
ap.add_argument("--patience", type=int, default=15)
ap.add_argument("--seed", type=int, default=0)
ap.add_argument("--out", default="results")
a = ap.parse_args()

torch.manual_seed(a.seed); np.random.seed(a.seed)
dev = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
d = Path(a.data); out = Path(a.out); out.mkdir(exist_ok=True)
classes = json.load(open(d / "classes.json"))
L = lambda s: (torch.from_numpy(np.load(d / f"{s}_X.npy")), torch.from_numpy(np.load(d / f"{s}_y.npy")))
(Xtr, ytr), (Xva, yva), (Xte, yte) = L("training"), L("validation"), L("test")

sample = Xtr[:1000].float()
mean = sample.mean(dim=(0, 2), keepdim=True)[0]            # (n_mels, 1) per-bin stats
std = sample.std(dim=(0, 2), keepdim=True)[0] + 1e-5
json.dump({"mean": mean.flatten().tolist(), "std": std.flatten().tolist()}, open(out / f"norm_{a.head}.json", "w"))
mean, std = mean.to(dev), std.to(dev)
prep = lambda x: ((x.to(dev).float() - mean) / std)


def augment(X, idx):
    """Random time crop per sample + one freq mask and one time mask."""
    T, crop = X.shape[2], min(a.crop, X.shape[2])
    starts = np.random.randint(0, T - crop + 1, size=len(idx))
    x = prep(torch.stack([X[j, :, s:s + crop] for j, s in zip(idx.tolist(), starts)]))
    for b in range(len(x)):
        f = np.random.randint(0, 16); f0 = np.random.randint(0, 128 - f + 1)
        t = np.random.randint(0, 24); t0 = np.random.randint(0, crop - t + 1)
        x[b, f0:f0 + f, :] = 0; x[b, :, t0:t0 + t] = 0
    return x


model = SongNet(n_classes=len(classes), head=a.head).to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=a.wd)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=4)


@torch.no_grad()
def predict(X):
    model.eval(); preds = []
    for i in range(0, len(X), 64):
        preds.append(song_log_probs(model(prep(X[i:i + 64]))).argmax(1).cpu())
    return torch.cat(preds).numpy()


best, bad = 0, 0
for ep in range(a.epochs):
    model.train(); perm = torch.randperm(len(Xtr)); tot = 0
    for i in range(0, len(perm), a.bs):
        idx = perm[i:i + a.bs]
        loss = F.nll_loss(song_log_probs(model(augment(Xtr, idx))), ytr[idx].to(dev))
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(idx)
    val = accuracy_score(yva.numpy(), predict(Xva)); sched.step(val)
    print(f"epoch {ep+1:02d} loss {tot/len(perm):.4f} val_acc {val:.4f}", flush=True)
    if val > best:
        best, bad = val, 0; torch.save(model.state_dict(), out / f"songnet_{a.head}.pt")
    else:
        bad += 1
        if bad >= a.patience: print("early stop"); break

model.load_state_dict(torch.load(out / f"songnet_{a.head}.pt", map_location=dev))
pred = predict(Xte); acc = accuracy_score(yte.numpy(), pred)
print(f"\nTEST accuracy ({a.head}): {acc:.4f}")
rep = classification_report(yte.numpy(), pred, target_names=classes, digits=3)
print(rep)
open(out / f"report_{a.head}.txt", "w").write(f"test_acc={acc:.4f}\n{rep}")
cm = confusion_matrix(yte.numpy(), pred)
plt.figure(figsize=(8, 6)); sns.heatmap(cm, annot=True, fmt="d", cmap="YlGnBu", xticklabels=classes, yticklabels=classes)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.tight_layout(); plt.savefig(out / f"confusion_{a.head}.png", dpi=150)

In [ ]:
%%writefile src/baselines.py
"""Classical baselines on FMA's precomputed librosa features (features.csv).
Official split:  python src/baselines.py --meta data/fma_metadata
Random split:    python src/baselines.py --meta data/fma_metadata --ids_dir data/processed_random --out results_random
"""
import argparse, json
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

ap = argparse.ArgumentParser()
ap.add_argument("--meta", required=True)
ap.add_argument("--out", default="results")
ap.add_argument("--ids_dir", default=None, help="use the split stored in this processed dir (e.g. data/processed_random)")
a = ap.parse_args()

tracks = pd.read_csv(Path(a.meta) / "tracks.csv", index_col=0, header=[0, 1])
feats = pd.read_csv(Path(a.meta) / "features.csv", index_col=0, header=[0, 1, 2])
small = tracks[tracks["set", "subset"] == "small"]
small = small[small["track", "genre_top"].notna()]


def get(split):
    if a.ids_dir:
        df = small.loc[np.load(Path(a.ids_dir) / f"{split}_ids.npy")]
    else:
        df = small[small["set", "split"] == split]
    return feats.loc[df.index].fillna(0).values, df["track", "genre_top"].values


Xtr, ytr = get("training"); Xte, yte = get("test")
print("train/test sizes:", len(Xtr), len(Xte))
sc = StandardScaler().fit(Xtr); Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)

models = {
    "KNN": KNeighborsClassifier(),
    "LogReg": LogisticRegression(max_iter=1000),
    "MLP": MLPClassifier(max_iter=300, random_state=0),
    "LinearSVM": LinearSVC(dual=False),
}
res = {}
for n, m in models.items():
    m.fit(Xtr, ytr); res[n] = float(accuracy_score(yte, m.predict(Xte)))
    print(f"{n}: {res[n]:.4f}", flush=True)
Path(a.out).mkdir(exist_ok=True)
json.dump(res, open(Path(a.out) / "baselines.json", "w"), indent=2)

In [ ]:
%%writefile src/resplit.py
"""Re-split the preprocessed FMA-small clips into a random, class-stratified 70/20/10
split (the protocol described in the SongNet paper) instead of FMA's official split.
Usage: python src/resplit.py --src data/processed --out data/processed_random
"""
import argparse, shutil
from pathlib import Path
import numpy as np
from sklearn.model_selection import train_test_split

ap = argparse.ArgumentParser()
ap.add_argument("--src", default="data/processed")
ap.add_argument("--out", default="data/processed_random")
ap.add_argument("--seed", type=int, default=0)
a = ap.parse_args()

S = ["training", "validation", "test"]
cat = lambda k: np.concatenate([np.load(Path(a.src) / f"{s}_{k}.npy") for s in S])
X, y, ids = cat("X"), cat("y"), cat("ids")
idx = np.arange(len(y))
tr, rest = train_test_split(idx, test_size=0.30, stratify=y, random_state=a.seed)
va, te = train_test_split(rest, test_size=1 / 3, stratify=y[rest], random_state=a.seed)

out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
for name, sel in zip(S, [tr, va, te]):
    np.save(out / f"{name}_X.npy", X[sel]); np.save(out / f"{name}_y.npy", y[sel])
    np.save(out / f"{name}_ids.npy", ids[sel]); print(name, len(sel))
shutil.copy(Path(a.src) / "classes.json", out / "classes.json")

## 5. Download FMA-small and metadata *(slow, ~7 GB)*
Source: https://github.com/mdeff/fma

In [ ]:
%cd /content/data
!wget -q --show-progress https://os.unil.cloud.switch.ch/fma/fma_metadata.zip
!wget -q --show-progress https://os.unil.cloud.switch.ch/fma/fma_small.zip
!unzip -q fma_metadata.zip
!unzip -q fma_small.zip
!ls
%cd /content

## 6. Preprocess audio into log-mel spectrograms *(slow)*
Many `mpg123` / `audioread` warnings are normal: a few FMA mp3s are corrupt and are skipped (about 3 of 6,400 training clips).

In [ ]:
!python src/data.py --meta data/fma_metadata --audio data/fma_small --out data/processed

## 7. Baselines on the official split
kNN, logistic regression, MLP, linear SVM on FMA's precomputed features.

In [ ]:
!python src/baselines.py --meta data/fma_metadata --out results

## 8. Create the paper-style random 70/20/10 split, then baselines on it

In [ ]:
!python src/resplit.py
!python src/baselines.py --meta data/fma_metadata --ids_dir data/processed_random --out results_random

## 9. Train SongNet on the random split *(slow)*
Paper-style protocol. Prints `epoch ... loss ... val_acc ...` per epoch and finishes with the test accuracy, per-genre report and confusion matrix (saved in `results_random/`).

Training stops early if validation accuracy does not improve for 15 epochs (max 80 epochs).

In [ ]:
!python src/train.py --data data/processed_random --out results_random --head dense

### GRU head (extension, genuine recurrent layer)

In [ ]:
!python src/train.py --data data/processed_random --out results_random --head gru

## 10. Train SongNet on the official split *(slow)*
Stricter, artist-separated evaluation. Results go to `results/`.

In [ ]:
!python src/train.py --head dense

In [ ]:
!python src/train.py --head gru

### Summarize results

In [ ]:
import json, glob
for d in ["results", "results_random"]:
    print("=" * 20, d, "=" * 20)
    try:
        print("baselines:", json.load(open(f"{d}/baselines.json")))
    except FileNotFoundError:
        print("no baselines yet")
    for f in sorted(glob.glob(f"{d}/report_*.txt")):
        print("\n", f); print(open(f).read())

In [ ]:
from IPython.display import Image, display
for f in ["results_random/confusion_dense.png", "results/confusion_dense.png"]:
    try: display(Image(f))
    except Exception as e: print(f, "not found")

## 11. Back up your work (do this before closing the tab!)
Colab erases `/content` when the session ends.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/songnet_backup
!tar czf /content/drive/MyDrive/songnet_backup/project.tgz --exclude=data/fma_small --exclude=data/fma_metadata --exclude='data/*.zip' src data results results_random
!ls -lh /content/drive/MyDrive/songnet_backup

In [ ]:
# Small download for your laptop (models, reports, class names; no big arrays)
!zip -r out.zip results results_random data/processed/classes.json
from google.colab import files
files.download("out.zip")

## 12. Run the demo on your laptop (not in Colab)
1. Unzip `out.zip` into your local project folder (keep `results/` and `data/processed/classes.json`).
2. `pip install -r requirements.txt` and install ffmpeg.
3. `streamlit run app.py` and open http://localhost:8501.

The demo needs `results/songnet_<head>.pt`, `results/norm_<head>.json` and `data/processed/classes.json`. To use the random-split model, copy the files from `results_random/` into `results/`.

## Reference numbers (paper)
Best baseline (SVM) ~46.4%; SongNet C-RNN ~65% (report 65.23%, poster 65.32%).